In [8]:
import json
import pathlib

import pandas as pd

In [9]:

project_root = pathlib.Path('../../..')
project_root.resolve()

PosixPath('/home/jnban/projects/roanoke-transit')

In [10]:
metroflex_data = project_root / 'data/metroflex'
metroflex_data.resolve()

PosixPath('/home/jnban/projects/roanoke-transit/data/metroflex')

In [13]:
with open(metroflex_data / 'metroflex-addresses.json', 'r') as f:
    address_locations = json.load(f)

lowercase_keys = {}
for k, v in address_locations.items():
    lowercase_keys[k.lower()] = v
for k, v in lowercase_keys.items():
    address_locations[k] = v
address_locations

{'Melrose Ave, Danbury, CT 06810': {'lat': 41.406647, 'lng': -73.434436},
 '1602 Hershberger Rd NW, Roanoke, VA 24012': {'lat': 37.314692,
  'lng': -79.961264},
 '1612 Hershberger Rd NW, Roanoke, VA 24012': {'lat': 37.314076,
  'lng': -79.961264},
 'Hardy, AR 72542': {'lat': 36.178441, 'lng': -91.481669},
 'Harriet, AR 72639': {'lat': 35.99341, 'lng': -92.52044},
 'Heart, AR 72539': {'lat': 36.323677, 'lng': -91.714867},
 '217 Westside Blvd NW, Apt 11, Roanoke, VA 24017': {'lat': 37.276689,
  'lng': -80.00188},
 '3301 Ordway Dr NW, Roanoke, VA 24017': {'lat': 37.308193, 'lng': -79.979194},
 '3305 Ordway Dr NW, Roanoke, VA 24017': {'lat': 37.308809, 'lng': -79.979495},
 'Ordway Dr NW, Roanoke, VA 24017': {'lat': 37.303432, 'lng': -79.97548},
 '1900 Braeburn Dr, Salem, VA 24153': {'lat': 37.26455, 'lng': -80.027873},
 '1901 Braeburn Dr, Salem, VA 24153': {'lat': 37.264834, 'lng': -80.028154},
 '1898 Braeburn Dr, Salem, VA 24153': {'lat': 37.263735, 'lng': -80.028278},
 '1902 Braeburn Dr,

In [74]:
[key for key in  lowercase_keys.keys() if key.startswith('2207')]

['2207 colonial ave sw, roanoke, va 24015',
 '2207 colonial ave, roanoke, va 24015']

In [76]:
def lookup(row, **kwargs):
    details = row['Pickup Address Details']
    
    val = row['Pickup Address']
    address, cityzip = val.lower().split('\n')
    
    
    if address.startswith('110 shenandoah ave') and details.lower().startswith('hotel roanoke'):
        matches = ['110 shenandoah ave ne, roanoke, va 24016']
    elif address == '2141 dale ave':
        matches = ['2141 dale ave se, roanoke, va 24013']
    elif address == '4176 franklin rd sw':
        matches = ['4176 franklin rd sw, roanoke, va 24014']
    elif address == '3162 williamson road':
        matches = ['3162 williamson rd ne, roanoke, va 24012']
    elif address == '378 elm ave sw  apt. #2':
        matches = ['378 elm ave sw, apt 2, roanoke, va 24016']
    elif address == '4340 electric road':
        matches = ['4340 electric rd, roanoke, va 24018']
    elif address == '1142 Jamison Ave Se APT. A'.lower():
        matches = ['1142 jamison ave se, apt a, roanoke, va 24013']
    elif address == '1906 BELLEVIEW AVE SE'.lower():
        matches = ['1906 belleview ave se, roanoke, va 24014']
    elif address == '601 Orange Ave Ne'.lower():
        matches = ['601 orange ave ne, roanoke, va 24013']
    elif address == '1320 East Washington Ave'.lower():
        matches = ['1320 e washington ave, vinton, va 24179']
    elif address == '650 N Jefferson St Apt 420'.lower():
        matches = ['650 n jefferson st, apt 420, roanoke, va 24016']
    elif address == '1015 7th St Se Apt B'.lower():
        matches = ['1015 7th st se, apt b, roanoke, va 24013']
    elif address == '2207 Colonial Ave'.lower():
        matches = ['2207 colonial ave, roanoke, va 24015']
    else:
        matches = [
            match
            for match in address_locations.keys()
            if match.startswith(address) and match.endswith(cityzip)
        ]
        
    if len(matches) == 0:
        raise ValueError(f'Address not found: {details} at {val}')
    if len(matches) == 1:
        return matches[0]
    raise ValueError(f'Multiple address locations found:{details} at {val} {matches}')

df = pd.read_csv(metroflex_data / 'metroflex-2025-02-trip-report-cleaned.csv')
for index, row in df.iterrows():
    df.loc[index, 'Pickup LonLat'] = lookup(row)
df[df['Pickup LonLat'].isnull()]

,Unnamed: 0,No show,Pickup Address Details,Dropoff Address Details,Guests,Pickup Address,Dropoff Address,Attendants,Trip Date,Pickup LonLat
228,602,No,ROSIE'S,NaN,0.0,"1135 Vinyard Road\nVinton, VA 24179","2609 Edison St Ne\nRoanoke, VA 24012",0.0,2025-02-02 11:39:30,None
230,606,No,ROSIE'S,BURGER KING,0.0,"1135 Vinyard Road\nVinton, VA 24179","716 Hardy Rd\nVinton, VA 24179",0.0,2025-02-09 12:57:38,None
233,612,No,ROSIE'S,BURGER KING,0.0,"1135 Vinyard Road\nVinton, VA 24179","716 Hardy Rd\nVinton, VA 24179",0.0,2025-02-16 15:34:39,None
235,618,No,NaN,green ridge baptis church,0.0,"5300 Hawthorne Rd Nw Apartment 617\nRoanoke, V...","5521 green ridge baptis church\nRoanoke, VA 24017",0.0,2025-01-26 09:16:18,None
236,620,No,green ridge baptis church,NaN,0.0,"5521 green ridge baptis church\nRoanoke, VA 24017","901 Clearwater Ave\nRoanoke, VA 24019",0.0,2025-01-26 12:02:40,None
...,...,...,...,...,...,...,...,...,...,...
1424,3888,No,CEI -- ROANOKE,NaN,0.0,"4411 Plantation Rd Ne\nRoanoke, VA 24019","1321 Leon St Nw\nRoanoke, VA 24017",0.0,2025-01-30 23:01:05,None
1425,3890,No,CEI -- ROANOKE,NaN,0.0,"4411 Plantation Rd Ne\nRoanoke, VA 24019","1321 Leon St Nw\nRoanoke, VA 24017",0.0,2025-01-31 22:49:07,None
1426,3892,No,CEI -- ROANOKE,NaN,0.0,"4411 Plantation Rd Ne\nRoanoke, VA 24019","1321 Leon St Nw\nRoanoke, VA 24017",0.0,2025-02-05 22:59:48,None
1427,3896,No,CEI -- ROANOKE,NaN,0.0,"4411 Plantation Rd Ne\nRoanoke, VA 24019","1321 Leon St Nw\nRoanoke, VA 24017",0.0,2025-02-13 23:12:26,None
